# Kohonen Self-Organizing Maps — Lecture Notebook

This notebook accompanies the lecture. It mirrors the web demos using the same algorithm in pure NumPy.

**Sections**
1. SOM class
2. Toy example: color organization
3. Evaluation: QE, TE, U-matrix
4. Image compression / decompression


## 1. The SOM class
We import the reference implementation from .

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from som import SOM, SOMConfig, image_to_blocks, blocks_to_image, compress_image, decompress, metrics
%matplotlib inline

## 2. Toy example: organizing random RGB colors

In [ ]:
rng = np.random.default_rng(0)
data = rng.random((2000, 3), dtype=np.float32)
cfg = SOMConfig(grid_h=30, grid_w=30, input_dim=3, n_iters=5000, seed=0)
som = SOM(cfg)
som.train(data, verbose=True)

plt.figure(figsize=(5,5))
plt.imshow(np.clip(som.weights, 0, 1))
plt.title('Trained SOM color map')
plt.axis('off')
plt.show()

## 3. Evaluation
Quantization Error (QE) and Topographic Error (TE), plus the U-matrix.

In [ ]:
qe = som.quantization_error(data)
te = som.topographic_error(data[:1000])
print(f'QE = {qe:.4f}   TE = {te:.4f}')

u = som.u_matrix()
plt.figure(figsize=(5,5))
plt.imshow(u, cmap='bone_r')
plt.title('U-matrix (bright = boundary)')
plt.colorbar(); plt.axis('off'); plt.show()

## 4. Image compression
Vector-quantize 4x4 RGB blocks using a 64-entry SOM codebook.

In [ ]:
from PIL import Image
payload, som_img = compress_image('sample.png', block=4, codebook=64, iters=1500)
rec = decompress(payload)
original = np.array(Image.open('sample.png').convert('RGB'))
m = metrics(original, rec, payload)
print(m)

fig, ax = plt.subplots(1, 2, figsize=(8,4))
ax[0].imshow(original); ax[0].set_title('Original'); ax[0].axis('off')
ax[1].imshow(rec); ax[1].set_title(f"Reconstructed (PSNR {m['PSNR_dB']:.2f} dB)"); ax[1].axis('off')
plt.show()

### Try it yourself
- Vary  (2, 4, 8) and  (16, 64, 256). What happens to PSNR? To compression ratio?
- Train longer (). Does QE keep dropping?
- Replace  with your own image.